# Load OSDU standard reference values into Iceberg

Use the same PySpark session/catalog as `osdu_well_wellbore_iceberg.sql`. Copy this notebook, `load_iceberg_reference_data.py`, and `AllReferenceValues.iceberg.xlsx` into one directory. The loader needs no repository checkout, manifest directory, or DDL file. Install `openpyxl` in the notebook environment if it is not already available.

The enriched workbook contains 51,924 complete reference records across all 47 prototype types, including supplementary units and inactive values. Unlike the four-column `AllReferenceValues.xlsx` inventory, it retains descriptions, relationships, conversion coefficients and attribution. JSON is split across cells when needed to avoid Excel truncation, and each record has an integrity checksum. To refresh the workbook from repository manifests, run `python Ontology\generate_iceberg_reference_workbook.py` from the repository root.

Fill in the five deployment settings below. Use the same namespace as your other OSDU records. ACL and legal metadata are stored, not enforced by these Iceberg tables. Retain the original providers' attribution and applicable licensing terms.

The first run validates all 47 target schemas and source records without writing. Review that output, then run the load cell. Existing IDs are skipped, not updated; run only one copy of this loader at a time. Writes are per table, not a transaction across the entire set. A serial rerun resumes after partial failure.

The 47 types are not a closed dependency set: references to types outside the prototype are preserved, not seeded. Records retain their original schema kinds, including older minor versions. Unmodeled properties are retained in `_raw_record`.

In [ ]:
%pip install --no-index --find-links="./wheels" openpyxl

In [ ]:
from pathlib import Path
import sys

spark.conf.set("spark.sql.debug.maxToStringFields", "2000")

# Directory containing the Python loader and enriched XLSX. Change if needed.
input_dir = Path.cwd()
if not (input_dir / 'load_iceberg_reference_data.py').is_file():
    input_dir = input_dir / 'Ontology'
sys.path.insert(0, str(input_dir))
from load_iceberg_reference_data import load_reference_data

settings = dict(
    workbook_path=input_dir / 'AllReferenceValues.iceberg.xlsx',
    catalog='osdu_iceberg',
    namespace='osdu',       # Your OSDU partition / record-ID namespace
    owners_group='owners@group.com',    # Your data-owner group
    viewers_group='vievers@group.com',   # Your data-viewer group
    legal_tag='sample_legal_tag_12345',       # Your deployment's legal tag
    country_code='US',    # Two-letter uppercase ISO country code
)

preview = load_reference_data(spark, **settings, dry_run=True)

In [ ]:
# Run only after the dry run has completed successfully.
spark.sparkContext.setLogLevel("ERROR")
report = load_reference_data(spark, **settings, dry_run=False)
print('Inserted:', sum(row['inserted'] for row in report))
print('Already present:', sum(row['already_present'] for row in report))

In [ ]:
catalog = settings['catalog']
spark.sql(f'''
    SELECT id, data.Code, data.Name, data.Description
    FROM `{catalog}`.reference_data.BasinType
    ORDER BY data.Code
''').show(30, truncate=False)

spark.sql(f'SELECT count(*) AS unit_count FROM `{catalog}`.reference_data.UnitOfMeasure').show()